# COMPASS multivariate_longitudinal models

Dynamic-DeepHit at every landmark, in two configurations for the selected
endpoint: a cause-only config (death censored, comparable to that endpoint's
Cox/XGBoost arms) and a competing config (cause + death as competing causes).

| `ENDPOINT` | configs run | inputs read |
|---|---|---|
| `platinum` | `platinum`, `competing` | `prediction_inputs_<arm>/` |
| `nepc` | `nepc`, `nepc_competing` | `prediction_inputs_<arm>_nepc/` |

Requires `01_preprocessing.ipynb` to have built the merged `profile_data`
inputs with `--build-longitudinal` (the default) **for that endpoint**. Set
`PREDICTION_INPUT_DIRS` in the setup cell to use inputs mounted at a different
location, such as a GPU cluster's shared filesystem.

**SurvLatent ODE is off by default** (`cp.RUN_SURVLATENT = False`). It needs
the bundled `survlatent_ode_repo/` checkout with its own conda env active; see
`multivariate_longitudinal/README.md`. Turn it on only once that environment
is available — it runs the same configs as Dynamic-DeepHit.

Kept separate from `03_multivariate.ipynb` so that notebook remains runnable
in a torch-free environment. This one requires torch.

In [ ]:
from pathlib import Path

ARMS = ["adt"]
# Which event to model. "platinum" reproduces the original run exactly.
# "nepc" runs the nepc/nepc_competing configs against the incident-NEPC
# cohort; it needs OUTPUT_SUFFIX = "_nepc" so it reads the NEPC prediction
# inputs and writes its own local_runs_adt_nepc/ tree. The two cohorts are
# NOT the same patients, so their metrics are not directly comparable.
ENDPOINT = "platinum"
OUTPUT_SUFFIX = ""  # "_nepc" when ENDPOINT == "nepc"

# SurvLatent ODE requires the bundled survlatent_ode_repo/ checkout with its
# conda env active (see multivariate_longitudinal/README.md). Left off so this
# notebook runs Dynamic-DeepHit alone.
RUN_SURVLATENT = False

# Optional per-arm prediction-input overrides for GPU jobs on another cluster.
# Leave this empty to use compass_pipeline.py's standard paths. Paths may be
# strings or Path objects and do not need to live under the repository.
# NOTE: these override the endpoint-suffixed defaults, so point them at the
# inputs matching ENDPOINT above.
PREDICTION_INPUT_DIRS = {
    # "adt": Path("/cluster/path/to/prediction_inputs_adt"),
    # "arpi": Path("/cluster/path/to/prediction_inputs_arpi"),
}

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.MAX_PRED_WINDOW = 260
cp.ENDPOINT = ENDPOINT
cp.RUN_SURVLATENT = RUN_SURVLATENT

if RUN_SURVLATENT:
    # Editable in-repo SurvLatent checkout. Override this only if a cluster
    # uses a different clone; the conda environment still needs to be active.
    cp.SURVLATENT_REPO = cp.DEFAULT_SURVLATENT_REPO

RUNS = cp.make_runs(
    ARMS, prediction_input_dirs=PREDICTION_INPUT_DIRS, output_suffix=OUTPUT_SUFFIX
)

print(f"endpoint={ENDPOINT}  configs={[c for _, c, _ in cp.longitudinal_task_specs(ENDPOINT)]}")

## Run multivariate_longitudinal models

Dynamic-DeepHit in both of the endpoint's configs (and SurvLatent ODE too, if
`RUN_SURVLATENT` was set above). Set `cp.FORCE_RERUN = False` to skip tasks
whose metrics file already exists. Layout:
`local_runs_<arm>[_nepc]/multivariate_longitudinal/<model>/landmark_{0,90,180}/<config>/`.

In [ ]:
for run in RUNS:
    cp.run_multivariate_longitudinal(run)

## Summary tables

Per-run C-index / mean AUC(t) / integrated Brier for every (model, landmark,
config), filtered to the endpoint's cause-of-interest row for headline
comparability against `03_multivariate.ipynb`'s Cox/XGBoost arms, then
combined across runs. The competing configs' death rows are written to disk
but excluded here — read them from the metrics CSVs directly if needed.

In [ ]:
summary_dfs = {run["label"]: cp.summarize_longitudinal_outputs(run) for run in RUNS}
for label, df in summary_dfs.items():
    print(f"=== {label} ===")
    display(df)

In [ ]:
import pandas as pd

combined_longitudinal_summary_df = pd.concat(summary_dfs.values(), ignore_index=True)
combined_longitudinal_summary_df